# Simulation study for Bayesian logistic regression example

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))  
sys.path.append(project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from algorithms import sig, log_p_laplace, soul_mh, soul_mh_decay, proximal_map_laplace_iterative, proximal_map_laplace_approx, mypipla, mypgd
#pip_ula, prox_pgd, proximal_map_laplace_iteration_total, pipgla, proximal_map_laplace_approx_total
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

os.chdir(project_root)

# Obtain synthetic dataset for Laplace prior case

In [ ]:
from scipy.stats import laplace, bernoulli
# Data and design matrix
np.random.seed(3)
design_matrix = np.random.uniform(low = -1.0, high = 1.0, size = (900, 50))
x_unknown = laplace.rvs(loc = -4, size = (50, 1))
parameter_bernoulli = sig(np.matmul(design_matrix, x_unknown))
data_experiment = bernoulli.rvs(np.array(parameter_bernoulli[:, 0]), size = 900)
labels = np.expand_dims(data_experiment, axis=1)
theta_true = -4

In [ ]:
# One run test
# --- SOUL Optimization Setup ---
D = 50       # Dimensionality of latent variables (x)
T = 800      # Outer optimization steps
M = 25       # MH steps per outer loop
B = 5       # Burn-in steps
delta_step = 0.08
proposal_std = 0.05
b_scale = 1.0 # Scale parameter for the Laplace prior

# Initializations
th0 = np.array([[-15.0]]) 
x0_M = np.zeros((D, 1))   # Initial latent variables vector

# Run the adapted algorithm
th_list, x_samples = soul_mh(
    log_p=log_p_laplace,
    th0=th0,
    x0_M=x0_M,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    D=D,
    delta_step=delta_step,
    proposal_std=proposal_std,
    b=b_scale
)

print("Final estimated theta:", th_list[-1][0, 0])
print("True theta (loc parameter): -4.0")

# SOUL Metropolis-Hastings algorithm iteratively

In [ ]:
# --- Experiment Parameters ---
T = 500  # Outer steps 
B = 5               
M = 20 

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.08    # Theta learning rate
proposal_std = 0.05  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale

fig = plt.figure(figsize=(10, 6))
theta_soul = []
X_soul = []

for run_idx in range(7):
    # Randomly initialize theta0 between -15 and 10
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])
    
    # Initialize X0 drawn from Gaussian centered at theta0, shape (D, M)
    # where D is feature dimension (50 from Paula's dataset)
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))
    
    th_list, x_values = soul_mh(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale
    )
    
    # Extract flattened array of theta trajectory across iterations
    th_trajectory = np.array([t[0, 0] for t in th_list])
    
    theta_soul.append(th_trajectory)
    X_soul.append(x_values)
    
    # Plot trajectory for this run
    plt.plot(th_trajectory, label=f'Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)')

# Reference line for ground truth mean
plt.axhline(y=np.mean(x_unknown), color='black', linestyle='dashed', label='True Mean $\\theta$')

plt.title('SOUL-MH Convergence Across Multiple Initializations')
plt.xlabel('Iteration (T)')
plt.ylabel('Theta Estimate ($\\theta$)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()
fig.savefig('soul_mh_convergence.pdf', format='pdf')

Because this loop was quite long to run, and all runs are independent, I tried to parallelise the runs:

In [ ]:
pip install joblib

In [ ]:
from joblib import Parallel, delayed

# tried parallelisation to go faster ! success :))

# --- Experiment Parameters ---
T = 1500  # Outer steps 
B = 5               
M = 20 

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.05    # Theta learning rate
proposal_std = 0.08  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale

def run_single_soul_mh(run_idx):
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))

    th_list, x_values = soul_mh(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale,
    )
    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values


# Run all 7 initializations in parallel across available CPU cores (leave at least one free so the computer stays responsive)
results = Parallel(n_jobs=-2)(
    delayed(run_single_soul_mh)(i) for i in range(7)
)

# Plot all 7 runs on one figure
fig = plt.figure(figsize=(10, 6))

theta_soul = []
X_soul = []

# Loop over the parallel results and add each line to the same plot
for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_soul.append(th_trajectory)
    X_soul.append(x_values)

    plt.plot(
        th_trajectory,
        label=f"Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)",
    )

# Reference line for ground truth
plt.axhline(
    y=np.mean(x_unknown),
    color="black",
    linestyle="dashed",
    label="True Mean $\\theta$",
)

plt.title("SOUL-MH Convergence Across Multiple Initializations (Parallel)")
plt.xlabel("Iteration (T)")
plt.ylabel("Theta Estimate ($\\theta$)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.show()
fig.savefig("soul_mh_convergence_parallel.pdf", format="pdf")

The relative error:

In [ ]:
list_relative_error = [
    np.abs(est - theta_true) / np.abs(theta_true) for est in list_map_estimate
]